In [1]:
import sys
sys.path.append('/home/kboone/software/Galaxy_Bias/Bispectrum/Sim_BiasFitting/')

import os
import jax
import h5py
import numpy as np
import jax.numpy as jnp
from Fits_Helpers import *
from scipy.optimize import least_squares
from scipy.interpolate import CubicSpline as spline

jax.config.update("jax_enable_x64", True)

In [2]:
z = 8
shift_kmax = 0.5
pos_root = '../../Data/Gala_z8_Positions/'
sim_root = '/mnt/nvme1/kboone/Data/BigFiles/Gala_8/'
model_root = '/mnt/nvme1/kboone/Sims/'

rescale = (8+1)/(z+1)
L = 2000
kF = 2*np.pi/L
kN_tsc = 2048*np.pi/L
kN = 256*np.pi/L
phases = 25

In [3]:
noises = np.zeros(phases, dtype=np.float64)
for ph in np.arange(phases):
    with h5py.File(pos_root + 'Gala_ph0' + '{:02d}'.format(ph) + '.h5', 'r') as f_:
        noises[ph] = L**3/f_['dataset'].shape[1]

In [4]:
def W(k):
    return (np.sin(np.pi*k/(2*kN_tsc)) / (np.pi*k/(2*kN_tsc)))**3

In [5]:
# Power spectrum data
numBins = 12
kMax_P = kN * shift_kmax

kBins = np.geomspace(kF, kMax_P, numBins + 1)
kBins = np.maximum(kBins, (np.arange(numBins+1)+0.5)*kF)
P_bins = []
for ph in np.arange(phases):
    power_file = sim_root + 'Gala_ph0' + '{:02d}'.format(ph) + '/Gala_256_ps.dat'
    P_bins_ind, kP, bin_idx, mask_P, N_P, N_P_tot = data_power(power_file, kBins, numBins)
    P_bins.append(P_bins_ind)
P_bins = np.array(P_bins)
std_P_bins = np.std(P_bins, axis=0, ddof=1)
Wk = W(kP)

# When multiplied by noises[ind] these will give the constant and k^2 power spectrum noise contributions.
Pnoise_c  = np.bincount(bin_idx[mask_P], weights=(Wk**2*N_P)[mask_P], minlength=numBins)/N_P_tot
Pnoise_k2 = np.bincount(bin_idx[mask_P], weights=(Wk**2*kP**2*N_P)[mask_P], minlength=numBins)/N_P_tot

In [6]:
# Bispectrum data
max_ksum = kN * shift_kmax
min_ksum = max_ksum/4

min_k = 0.5*kF * shift_kmax
max_k = 64.5*kF * shift_kmax

Bs = []
for ph in np.arange(phases):
    bs_file = sim_root + 'Gala_ph0' + '{:02d}'.format(ph) + '/Gala_256_comb.dat'
    B, kB = data_bispec(bs_file, min_k, max_k, min_ksum, max_ksum)
    Bs.append(B)
Bs = np.array(Bs)
std_B = np.std(Bs, axis=0, ddof=1)
k1, k2, k3 = kB[:,0], kB[:,1], kB[:,2]
W123 = W(k1)*W(k2)*W(k3)

# Number of bins for the power spectrum and bispectrum respectively.
N_Pb, N_Bb = numBins, Bs.shape[1]

In [7]:
# name: (sym_factor, rescale power)
P_files = {'b1b1': (1, 2), 'F2F2': (1, 4), 'b1F3': (2, 4), 'b2b2': (1, 4), 'b2F2': (2, 4),
           'b2K2': (2, 4), 'K2K2': (1, 4), 'K2F2': (2, 4), 'b1K2_2': (2, 4), 'b1Otd': (2, 4),
           'b1bk2': (-2, 2), 'b3b3': (1, 6)}
B_files = {'b1b1b2': (3, 4), 'b1b1K2': (3, 4), 'b1b1F2': (3, 4), 'b2b2b2': (1, 6), 'b1b2b3': (6, 6)}

P_tmpl, P_sum, B_tmpl = {}, {}, {}
for name, (sym, rpow) in P_files.items():
    unbinned, binned = sims_power(model_root + 'PowerSpectra/' + name + '_ps.dat', kBins, numBins, Wk, sym_factor=sym)
    P_tmpl[name] = binned * rescale**rpow
    S = spline(kP, unbinned * rescale**rpow, bc_type='natural')
    P_sum[name] = S(k1) + S(k2) + S(k3)
for name, (sym, rpow) in B_files.items():
    B_tmpl[name] = sims_bispec(model_root + 'Bispectra/' + name + '_comb.dat', kB, sym_factor=sym) * W123 * rescale**rpow

In [8]:
# # Laplacian sign check: bk2 multiplies the Laplacian of delta, which is -k^2 delta, so with sym_factor -2 the
# # b1bk2 template should be exactly -2 k^2 P_11 (b1b1) at every k.
# import matplotlib.pyplot as plt
# P_lap = sims_power(model_root + 'PowerSpectra/b1bk2_ps.dat', kBins, numBins, Wk, sym_factor=-2)[0]
# P_lin = sims_power(model_root + 'PowerSpectra/b1b1_ps.dat', kBins, numBins, Wk, sym_factor=1)[0]
# plt.semilogx(kP[mask_P], (P_lap/(kP**2*P_lin))[mask_P], '.', ms=2)
# plt.axhline(-2, c='k', ls='--'); plt.xlabel('k [h/Mpc]'); plt.ylabel(r'$P_{b_1 b_{k^2}} / (k^2 P_{11})$');

In [9]:
# Leave one out inverse covariance for the power spectrum
def C_inv_P_loo(ind):
    P_oth = np.delete(P_bins, ind, axis=0)
    M = len(P_oth)
    C_P = np.cov(P_oth/std_P_bins, rowvar=False)
    return np.linalg.inv(C_P) * (M - numBins - 2) / (M - 1)

## Bispectrum covariance

In [10]:
# Leave one out inverse variances for the bispectrum (diagonal only), in units of std_B.
# One-bin Hartlap correction (M-3)/(M-1).
def C_inv_B_loo(ind):
    B_oth = np.delete(Bs, ind, axis=0)
    M = len(B_oth)
    return (std_B / np.std(B_oth, axis=0, ddof=1))**2 * (M - 3) / (M - 1)


# R with R.T @ R = M, so that c^T M c = |R c|^2 (needed for least_squares).
def sqrt_psd(M):
    w, V = np.linalg.eigh(M)
    return np.sqrt(np.clip(w, 0, None))[:, None] * V.T

## Model contributions

In [11]:
# Power spectrum contributions                    Bispectrum contributions
#   idx  template   bias coefficient                  idx  template   bias coefficient
#   0    b1b1       b1^2                              0    b1b1b2     b1^2 b2
#   1    F2F2       b1^2                              1    b1b1K2     b1^2 bK2
#   2    b1F3       b1^2                              2    b1b1F2     b1^3
#   3    b2b2       b2^2                              3    b2b2b2     b2^3
#   4    b2F2       b1 b2                             4    b1b2b3     b1 b2 b3
#   5    b2K2       b2 bK2                            5    constant noise   bBCN
#   6    K2K2       bK2^2
#   7    K2F2       b1 bK2                         The bispectrum shot noise bBPN * sum_P(k1)+P(k2)+P(k3)
#   8    b1K2_2     b1 bK2                         uses whichever of P contributions 0-11 are kept.
#   9    b1Otd      b1 bOtd
#   10   b1bk2      b1 bk2
#   11   b3b3       b3^2
#   12   constant noise   bPN
#   13   k^2 noise        bPk2N
P_names = ['b1b1', 'F2F2', 'b1F3', 'b2b2', 'b2F2', 'b2K2', 'K2K2', 'K2F2', 'b1K2_2', 'b1Otd', 'b1bk2', 'b3b3']
B_names = ['b1b1b2', 'b1b1K2', 'b1b1F2', 'b2b2b2', 'b1b2b3']
param_names = ['bPN', 'bPk2N', 'bBCN', 'bBPN', 'b1', 'b2', 'bK2', 'bOtd', 'bk2', 'b3']

# option: (P indexes to drop, B indexes to drop, parameters fixed to 0)
DROP = {'no_k2_noise':  ([13],            [],        ['bPk2N']),
        'no_nl_matter': ([1, 2, 4, 7, 8], [2],       []),
        'no_tidal':     ([5, 6, 7, 8],    [1],       ['bK2']),
        'no_btd':       ([9],             [],        ['bOtd']),
        'no_blap':      ([10],            [],        ['bk2']),
        'no_b3':        ([11],            [3, 4],    ['b3']), # No b3 also for me means dropping to tree level bispectrum
        'no_b2':        ([3, 4, 5],       [0, 3, 4], ['b2'])}

In [12]:
# Only change the variables in this block from run to run.
# Full: only no_b3. Full + b3: nothing. Linear matter: no_b3, no_nl_matter.
# Alt: no_nl_matter, no_tidal, no_btd, no_blap. Simple: Alt + no_k2_noise.
model = dict(no_k2_noise=False, # DO NOT CHANGE THIS FROM FALSE
             no_nl_matter=False, no_tidal=False, 
             no_btd=False, no_blap=False, no_b3=True, no_b2=False)
write_root = 'Shift_kmax/EFT_'
os.makedirs(os.path.dirname(write_root), exist_ok=True)

on = [DROP[key] for key, val in model.items() if val]
drop_P = set().union(*[d[0] for d in on])
drop_B = set().union(*[d[1] for d in on])
fixed = set().union(*[d[2] for d in on])

keep_P = np.array([i for i in range(14) if i not in drop_P])
keep_B = np.array([i for i in range(6) if i not in drop_B])
keep_PS = keep_P[keep_P < 12]
free = np.array([i for i, name in enumerate(param_names) if name not in fixed])
print('Free parameters:', [param_names[i] for i in free])

Free parameters: ['bPN', 'bPk2N', 'bBCN', 'bBPN', 'b1', 'b2', 'bK2', 'bOtd', 'bk2']


## Fitting

In [13]:
def fit(ind):
    # Check out how this works.
    C_inv_P = C_inv_P_loo(ind)
    C_inv_B = C_inv_B_loo(ind)

    P_terms = np.array([P_tmpl[name] for name in P_names]
                       + [noises[ind]*Pnoise_c,
                          noises[ind]*Pnoise_k2])[keep_P]
    P_terms = np.vstack([P_terms, -P_bins[ind]])/std_P_bins
    P_chi2_matrix = P_terms @ C_inv_P @ P_terms.T

    B_terms = np.array([B_tmpl[name] for name in B_names]
                       + [W123*noises[ind]**2])[keep_B]
    B_noise = np.array([W123*noises[ind]*P_sum[name] for name in P_names])[keep_PS]
    B_terms = np.vstack([B_terms, B_noise, -Bs[ind]])/std_B
    B_chi2_matrix = (B_terms*C_inv_B) @ B_terms.T

    R_P = jnp.asarray(sqrt_psd(P_chi2_matrix))
    R_B = jnp.asarray(sqrt_psd(B_chi2_matrix))

    def bias_terms(params):
        bPN, bPk2N, bBCN, bBPN, b1, b2, bK2, bOtd, bk2, b3 = params
        P_bias_terms = jnp.array([b1**2, b1**2, b1**2, b2**2, b1*b2, b2*bK2, bK2**2,
                                  b1*bK2, b1*bK2, b1*bOtd, b1*bk2, b3**2, bPN, bPk2N])
        B_bias_terms = jnp.array([b1**2*b2, b1**2*bK2, b1**3, b2**3, b1*b2*b3, bBCN])
        P_vec = jnp.append(P_bias_terms[keep_P], 1.0)
        B_vec = jnp.concatenate([B_bias_terms[keep_B], bBPN*P_bias_terms[keep_PS], jnp.ones(1)])
        return P_vec, B_vec

    def _residuals(free_params):
        params = jnp.zeros(10).at[free].set(free_params)
        P_vec, B_vec = bias_terms(params)
        return jnp.concatenate([R_P @ P_vec, R_B @ B_vec])

    residuals = jax.jit(_residuals)
    jacobian = jax.jit(jax.jacfwd(_residuals))

    # Starting point depends on redshift (bias is much smaller at z = 5).
    if z == 8:
        ind_initial_guess = np.array([1.0, 1.0, 1.0, 1.0, 10.0, 40.0, 1.0, 10.0, 1.0, 100.0])
    elif z == 5:
        ind_initial_guess = np.array([1.0, 1.0, 1.0, 1.0, 5.0, 6.0, -1.0, 1.0, 1.0, 10.0])

    sol = least_squares(lambda x: np.asarray(residuals(x)), ind_initial_guess[free], 
                          jac=lambda x: np.asarray(jacobian(x)), method='trf', x_scale='jac', 
                          xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=5000)

    ind_best_params = np.zeros(10)
    ind_best_params[free] = sol.x
    P_vec, B_vec = (np.asarray(v) for v in bias_terms(ind_best_params))
    ind_chi2 = np.array([P_vec @ P_chi2_matrix @ P_vec, B_vec @ B_chi2_matrix @ B_vec])
    return ind_best_params, ind_chi2

## Run

In [14]:
best_params = []
chi2 = []
for ind in np.arange(phases):
    best_params_ind, chi2_ind = fit(ind)
    best_params.append(best_params_ind)
    chi2.append(chi2_ind)
best_params = np.array(best_params)
chi2 = np.array(chi2)

# Best fits: all 10 parameters in param_names order (fixed ones are 0). Chi2: (chi2_P, chi2_B) per phase.
np.save(write_root + 'Best_Fits.npy', best_params)
# np.save(write_root + 'Chi2.npy', chi2)

dof = N_Pb + N_Bb - len(free)
print(f'chi2_P/N_P = {chi2[:,0].mean()/N_Pb:.3f}   chi2_B/N_B = {chi2[:,1].mean()/N_Bb:.3f}   '
      f'total = {chi2.sum(1).mean():.1f}   total/dof = {chi2.sum(1).mean()/dof:.3f}')

chi2_P/N_P = 0.441   chi2_B/N_B = 0.943   total = 361.9   total/dof = 0.950


## Cross-validation

In [15]:
# Cross-validation: score each phase's best-fit bias parameters (from the Run cell) on every phase's data.
# cv_chi2[i, j] = (chi2_P, chi2_B) of phase i's best-fit bias parameters (b1, b2, ...) on phase j's data, using
# phase j's shot noise and leave-one-out covariance, with the free noise parameters (bPN, bPk2N | bBCN, bBPN)
# refit on phase j. The diagonal (i = j) is the usual training chi2; the off-diagonal entries are test chi2.
# With the bias parameters fixed, P_vec and B_vec are linear in the noise parameters, so chi2 is quadratic
# in them and the refit is an exact linear solve.
# To compare two models, match the off-diagonal entries elementwise (Model_Comparison.ipynb).

# Same chi2 matrices as inside fit(), for phase j.
def chi2_matrices(j):
    P_terms = np.array([P_tmpl[name] for name in P_names]
                       + [noises[j]*Pnoise_c,
                          noises[j]*Pnoise_k2])[keep_P]
    P_terms = np.vstack([P_terms, -P_bins[j]])/std_P_bins
    P_chi2_matrix = P_terms @ C_inv_P_loo(j) @ P_terms.T

    B_terms = np.array([B_tmpl[name] for name in B_names]
                       + [W123*noises[j]**2])[keep_B]
    B_noise = np.array([W123*noises[j]*P_sum[name] for name in P_names])[keep_PS]
    B_terms = np.vstack([B_terms, B_noise, -Bs[j]])/std_B
    B_chi2_matrix = (B_terms*C_inv_B_loo(j)) @ B_terms.T
    return P_chi2_matrix, B_chi2_matrix

# Same as bias_terms() inside fit(), in numpy.
def bias_vecs(params):
    bPN, bPk2N, bBCN, bBPN, b1, b2, bK2, bOtd, bk2, b3 = params
    P_bias_terms = np.array([b1**2, b1**2, b1**2, b2**2, b1*b2, b2*bK2, bK2**2,
                             b1*bK2, b1*bK2, b1*bOtd, b1*bk2, b3**2, bPN, bPk2N])
    B_bias_terms = np.array([b1**2*b2, b1**2*bK2, b1**3, b2**3, b1*b2*b3, bBCN])
    P_vec = np.append(P_bias_terms[keep_P], 1.0)
    B_vec = np.concatenate([B_bias_terms[keep_B], bBPN*P_bias_terms[keep_PS], np.ones(1)])
    return P_vec, B_vec

mats = [chi2_matrices(j) for j in range(phases)]
noise_idx = [[a for a in (0, 1) if a in free], [a for a in (2, 3) if a in free]]  # P, B

cv_chi2 = np.zeros((phases, phases, 2))
for i in range(phases):
    params0 = best_params[i].copy()
    params0[:4] = 0
    vecs0 = bias_vecs(params0)
    for s in range(2):  # 0: power spectrum, 1: bispectrum
        # Column a of U: change in the bias vector per unit of noise parameter a.
        U = np.array([bias_vecs(np.where(np.arange(10) == a, 1.0, params0))[s] - vecs0[s]
                      for a in noise_idx[s]]).T
        for j in range(phases):
            M = mats[j][s]
            v = vecs0[s] - U @ np.linalg.solve(U.T @ M @ U, U.T @ M @ vecs0[s]) if len(noise_idx[s]) else vecs0[s]
            cv_chi2[i, j, s] = v @ M @ v
assert np.allclose(cv_chi2[np.arange(phases), np.arange(phases)], chi2)  # diagonal reproduces the fits

np.save(write_root + 'CV_Chi2.npy', cv_chi2)

# Mean test chi2 per fit phase i (average over the 24 other phases j).
off = ~np.eye(phases, dtype=bool)
test = cv_chi2.sum(2)[off].reshape(phases, phases - 1).mean(1)
print(f'train chi2 = {chi2.sum(1).mean():.1f}   test chi2 = {test.mean():.1f}   '
      f'(test - train = {(test - chi2.sum(1)).mean():.1f})')

train chi2 = 361.9   test chi2 = 380.8   (test - train = 19.0)


In [16]:
np.sum(np.sum(cv_chi2, axis=2)[off]) / np.sum(off)

np.float64(380.8427784574109)

### Cross-validation, slow version (one pair at a time)
Same as `fit()`, but scored on phase `ind_j`'s data with the bias parameters (b1, b2, ...) locked to phase `ind_i`'s best fit.
Only the noise parameters (bPN, bPk2N, bBCN, bBPN) are free, and they are found with the same `least_squares` call
instead of the linear solve above. `cv_chi2[ind_i, ind_j]` should match.

In [17]:
def cv_fit(ind_i, ind_j):
    # Phase j's data, shot noise and leave-one-out covariance, exactly as in fit(ind_j).
    C_inv_P = C_inv_P_loo(ind_j)
    C_inv_B = C_inv_B_loo(ind_j)

    P_terms = np.array([P_tmpl[name] for name in P_names]
                       + [noises[ind_j]*Pnoise_c,
                          noises[ind_j]*Pnoise_k2])[keep_P]
    P_terms = np.vstack([P_terms, -P_bins[ind_j]])/std_P_bins
    P_chi2_matrix = P_terms @ C_inv_P @ P_terms.T

    B_terms = np.array([B_tmpl[name] for name in B_names]
                       + [W123*noises[ind_j]**2])[keep_B]
    B_noise = np.array([W123*noises[ind_j]*P_sum[name] for name in P_names])[keep_PS]
    B_terms = np.vstack([B_terms, B_noise, -Bs[ind_j]])/std_B
    B_chi2_matrix = (B_terms*C_inv_B) @ B_terms.T

    R_P = jnp.asarray(sqrt_psd(P_chi2_matrix))
    R_B = jnp.asarray(sqrt_psd(B_chi2_matrix))

    def bias_terms(params):
        bPN, bPk2N, bBCN, bBPN, b1, b2, bK2, bOtd, bk2, b3 = params
        P_bias_terms = jnp.array([b1**2, b1**2, b1**2, b2**2, b1*b2, b2*bK2, bK2**2,
                                  b1*bK2, b1*bK2, b1*bOtd, b1*bk2, b3**2, bPN, bPk2N])
        B_bias_terms = jnp.array([b1**2*b2, b1**2*bK2, b1**3, b2**3, b1*b2*b3, bBCN])
        P_vec = jnp.append(P_bias_terms[keep_P], 1.0)
        B_vec = jnp.concatenate([B_bias_terms[keep_B], bBPN*P_bias_terms[keep_PS], jnp.ones(1)])
        return P_vec, B_vec

    # The only change from fit(): start from phase i's best fit and let just the free noise parameters vary.
    locked_params = best_params[ind_i].copy()
    vary = np.array([a for a in free if a < 4])  # bPN, bPk2N, bBCN, bBPN (whichever are free)

    def _residuals(vary_params):
        params = jnp.asarray(locked_params).at[vary].set(vary_params)
        P_vec, B_vec = bias_terms(params)
        return jnp.concatenate([R_P @ P_vec, R_B @ B_vec])

    residuals = jax.jit(_residuals)
    jacobian = jax.jit(jax.jacfwd(_residuals))

    sol = least_squares(lambda x: np.asarray(residuals(x)), locked_params[vary],
                          jac=lambda x: np.asarray(jacobian(x)), method='trf', x_scale='jac',
                          xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=5000)

    ij_params = locked_params.copy()
    ij_params[vary] = sol.x
    P_vec, B_vec = (np.asarray(v) for v in bias_terms(ij_params))
    ij_chi2 = np.array([P_vec @ P_chi2_matrix @ P_vec, B_vec @ B_chi2_matrix @ B_vec])
    return ij_params, ij_chi2

ind_i, ind_j = 0, 1
ij_params, ij_chi2 = cv_fit(ind_i, ind_j)
print('params:', dict(zip(param_names, ij_params)))
print('chi2 (P, B):', ij_chi2, '   fast version:', cv_chi2[ind_i, ind_j])

params: {'bPN': np.float64(1.4716668934843173), 'bPk2N': np.float64(-6.775679807886525), 'bBCN': np.float64(-5.597374714266961), 'bBPN': np.float64(4.396421831407432), 'b1': np.float64(10.290678768716164), 'b2': np.float64(13.155965917673955), 'bK2': np.float64(-8.424848440527871), 'bOtd': np.float64(190.9964936395546), 'bk2': np.float64(-126.38739812223476), 'b3': np.float64(0.0)}
chi2 (P, B): [  5.72187217 373.31664052]    fast version: [  5.72187217 373.31664052]
